## Run on Google Colab

In [2]:
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# Load the dataset
ds = load_dataset("edaschau/bitcoin_news")
df = ds['train'].to_pandas()

# Data filtering
df['date'] = pd.to_datetime(df['time_unix'], unit="s")
df['date'] = df['date'].dt.ceil('h')
df = df.sort_values(by='date', ascending=True)
df.set_index('date', inplace=True)
df = df.loc['2018-05-15 06:00:00':'2025-12-31']
df = df[['title', 'article_text', 'url']]

df.head(3)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,title,article_text,url
date,,,
2018-05-15 07:00:00,Project Jasper Reveals Proof of Concept for Ac...,"jasper.jpg Payments Canada, the Bank of Canada...",https://finance.yahoo.com/news/project-jasper-...
2018-05-15 07:00:00,Dubai Bitcoin Exchange Halts Fiat Withdrawals ...,Dubai-based exchange BitOasis has suspended UA...,https://finance.yahoo.com/news/dubai-bitcoin-e...
2018-05-15 08:00:00,Bitcoin Prices Advance; South Korea to Widen P...,Bitcoin prices advanced on Tuesday Investing.c...,https://finance.yahoo.com/news/bitcoin-prices-...


In [3]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")
model = AutoModelForSequenceClassification.from_pretrained("mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")

In [4]:
def analyze_sentiment_in_batches(texts, batch_size):
    sentiment_labels = []

    for i in tqdm(range(0, len(texts), batch_size)):  # Live progress bar
        batch_texts = texts[i:i + batch_size]

        # Tokenize and move inputs to GPU
        inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, max_length=512, padding=True)
        inputs = {key: val.to(device) for key, val in inputs.items()}  # Move tensors to GPU

        # Perform inference on GPU
        with torch.no_grad():
            outputs = model(**inputs)

        # Get predicted class labels
        predicted_classes = torch.argmax(outputs.logits, dim=1)
        batch_labels = [model.config.id2label[idx.item()] for idx in predicted_classes]
        sentiment_labels.extend(batch_labels)

    return sentiment_labels

# Ensure model and device are set correctly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

# Run with live progress bar
df['sentiment_labels'] = analyze_sentiment_in_batches(df['article_text'].tolist(), batch_size=256)

Using device: cuda


100%|██████████| 270/270 [19:53<00:00,  4.42s/it]


In [8]:
df.head(3)

,title,article_text,url,sentiment_labels
date,,,,
2018-05-15 07:00:00,Project Jasper Reveals Proof of Concept for Ac...,"jasper.jpg Payments Canada, the Bank of Canada...",https://finance.yahoo.com/news/project-jasper-...,positive
2018-05-15 07:00:00,Dubai Bitcoin Exchange Halts Fiat Withdrawals ...,Dubai-based exchange BitOasis has suspended UA...,https://finance.yahoo.com/news/dubai-bitcoin-e...,neutral
2018-05-15 08:00:00,Bitcoin Prices Advance; South Korea to Widen P...,Bitcoin prices advanced on Tuesday Investing.c...,https://finance.yahoo.com/news/bitcoin-prices-...,positive


In [7]:
# Save the dataset to a CSV file in Google Drive
from google.colab import drive
drive.mount('/content/drive')
df.to_csv('/content/drive/My Drive/bitcoin_news.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
